In [1]:
import pandas as pd
from pipeline import create_1_week_X_y_df, create_2_week_X_y_df, create_3_week_X_y_df


from features.Carl.time_features import usage_metrices, rolling_mean_length_per_userId

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import datetime


In [2]:
df_full = pd.read_parquet("../../../data/churn-prediction-25-26/train.parquet")

It makes the most sense to focus on the time usage features.
Most importantly the total time usage of the application. 


In [3]:
df = df_full.copy()

In [4]:
df_X_slices, df_y_slices = create_2_week_X_y_df(df = df)

In [5]:

# Idea add weekday and weeknumber as input and for each user
# compute their usage within the week and on this day
#


# Goal have a metric:
# Mean usage
# Mean usage this week
# mean usage last week
# mean usage during week
# mean usage on the weekend

# Drop in usage in percentage this week compared to last week

In [6]:
df_X = df_X_slices[0].copy()

In [7]:
df_X_length_rolling_mean = rolling_mean_length_per_userId(df = df_X, rolling_mean_window=3)

In [8]:
# weekly average drop in usage

df_X_length_rolling_mean["week"] = pd.to_datetime(df_X_length_rolling_mean["date"]).dt.isocalendar().week

In [9]:
df_X_length_rolling_mean.sample(10)

df_a = df_X_length_rolling_mean.groupby(["userId", "week"])["rolling_avg"].sum().reset_index()

#df_a = df_X_length_rolling_mean.pivot(index = "userId", columns = "week", values = "rolling_avg")

In [10]:
week_col = sorted(df_a["week"].unique())

df_b = df_a.pivot(index = "userId", columns="week", values = "rolling_avg")

In [11]:
delta_cols = []
delta_cols_per = []
# compute deltas
for index in range(len(week_col) - 1):
    df_b[f"delta_week_{index}_{index+1}"] = df_b[week_col[index + 1]] - df_b[week_col[index]]

    df_b[f"delta_week_{index}_{index+1}_%"] = ((df_b[week_col[index + 1]] - df_b[week_col[index]]) / df_b[week_col[index]]) * 100

    df_b[f"delta_week_{index}_{index+1}_%"] = df_b[f"delta_week_{index}_{index+1}_%"].replace(np.inf, 0.0)

    df_b[f"delta_week_{index}_{index+1}_%"] = df_b[f"delta_week_{index}_{index+1}_%"].fillna(0.0)

    delta_cols.append(f"delta_week_{index}_{index+1}")
    delta_cols_per.append(f"delta_week_{index}_{index+1}_%")

df_b

week,40,41,42,delta_week_0_1,delta_week_0_1_%,delta_week_1_2,delta_week_1_2_%
userId,,,,,,,
1000025,226346.156860,90966.789783,13152.134360,-135379.367077,-59.810765,-77814.655423,-85.541829
1000035,625.631290,16115.484900,5070.548787,15489.853610,2475.875785,-11044.936113,-68.536170
1000103,9769.046820,0.000000,0.000000,-9769.046820,-100.000000,0.000000,0.000000
1000164,408.702527,29429.104060,894.161360,29020.401533,7100.617109,-28534.942700,-96.961643
1000168,19520.589290,38769.588940,0.000000,19248.999650,98.608702,-38769.588940,-100.000000
...,...,...,...,...,...,...,...
1999781,107057.285747,26745.327660,9643.502303,-80311.958087,-75.017742,-17101.825357,-63.943226
1999847,9996.154460,12378.855980,0.000000,2382.701520,23.836181,-12378.855980,-100.000000
1999848,27023.025327,17617.539040,0.000000,-9405.486287,-34.805453,-17617.539040,-100.000000


In [12]:
delta_cols

df_b["delta_variance"] = df_b[delta_cols].var(axis = 1)
df_b["delta_mean"] = df_b[delta_cols].mean(axis = 1)

df_b["delta_variance_%"] = df_b[delta_cols_per].var(axis = 1)
df_b["delta_mean_%"] = df_b[delta_cols_per].mean(axis = 1)

In [13]:
df_b.columns = df_b.columns.astype(str)

In [14]:
df_b.columns

Index(['40', '41', '42', 'delta_week_0_1', 'delta_week_0_1_%',
       'delta_week_1_2', 'delta_week_1_2_%', 'delta_variance', 'delta_mean',
       'delta_variance_%', 'delta_mean_%'],
      dtype='object', name='week')

In [15]:
for i, week in enumerate(week_col):
    df_b = df_b.rename(columns={
        str(week) : i
    })

In [17]:
df_b = df_b.reset_index()

In [18]:
df_b

week,userId,0,1,2,delta_week_0_1,delta_week_0_1_%,delta_week_1_2,delta_week_1_2_%,delta_variance,delta_mean,delta_variance_%,delta_mean_%
0,1000025,226346.156860,90966.789783,13152.134360,-135379.367077,-59.810765,-77814.655423,-85.541829,1.656848e+09,-106597.011250,3.310438e+02,-72.676297
1,1000035,625.631290,16115.484900,5070.548787,15489.853610,2475.875785,-11044.936113,-68.536170,3.520475e+08,2222.458748,3.237016e+06,1203.669807
2,1000103,9769.046820,0.000000,0.000000,-9769.046820,-100.000000,0.000000,0.000000,4.771714e+07,-4884.523410,5.000000e+03,-50.000000
3,1000164,408.702527,29429.104060,894.161360,29020.401533,7100.617109,-28534.942700,-96.961643,1.656309e+09,242.729417,2.590257e+07,3501.827733
4,1000168,19520.589290,38769.588940,0.000000,19248.999650,98.608702,-38769.588940,-100.000000,1.683078e+09,-9760.294645,1.972271e+04,-0.695649
...,...,...,...,...,...,...,...,...,...,...,...,...
14915,1999781,107057.285747,26745.327660,9643.502303,-80311.958087,-75.017742,-17101.825357,-63.943226,1.997760e+09,-48706.891722,6.132245e+01,-69.480484
14916,1999847,9996.154460,12378.855980,0.000000,2382.701520,23.836181,-12378.855980,-100.000000,1.089518e+08,-4998.077230,7.667700e+03,-38.081909
14917,1999848,27023.025327,17617.539040,0.000000,-9405.486287,-34.805453,-17617.539040,-100.000000,3.371891e+07,-13511.512663,2.125165e+03,-67.402726
14918,1999892,14715.750687,9391.282093,0.000000,-5324.468593,-36.182107,-9391.282093,-100.000000,8.269486e+06,-7357.875343,2.036362e+03,-68.091053
